In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib
import shap
from math import sqrt
from keras.callbacks import ModelCheckpoint
from keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from imblearn.over_sampling import SMOTE
from sklearn import svm
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from matplotlib import font_manager, rc
path = "c:/Windows/Fonts/malgun.ttf"
if platform.system() == 'Darwin':
    rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    font_name = font_manager.FontProperties(fname=path).get_name()
    rc('font', family=font_name)
else:
    print('Unknown system... sorry~~~~')

C:\Anaconda3\lib\site-packages\statsmodels\tools\_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm
Using TensorFlow backend.


In [3]:
data = pd.read_csv("C:/Users/PC/Desktop/데이터셋_005_민재2.csv")
data.head()

Columns (60) have mixed types.Specify dtype option on import or set low_memory=False.


,연번,시도명,시군구명,관리기관,단지명,입주일,최초등록일,기준년월,노동지청,행정구역,...,평균급여액_2016,평균급여액_2017,평균급여액_2018,평균급여액_2019,평균급여액_2020,평균급여액_2021,평균급여액_2022,위도,경도,재해발생년도
0,164,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20180604.0,NaN,202012,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.481999,126.897686,2019
1,719,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20100202.0,20210419.0,201812,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482491,126.894409,2018
2,868,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,NaN,19970930.0,201912,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482035,126.897731,2019
3,892,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20060119.0,20060703.0,201912,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482128,126.895313,2018
4,979,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20030626.0,20111130.0,202112,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482491,126.894409,2021


In [4]:
data['기업규모'] = 0
data.loc[data['근로자수'] >= 5, '기업규모'] = 1
data.loc[data['근로자수'] >= 50, '기업규모'] = 2
data.loc[data['근로자수'] >= 100, '기업규모'] = 3
data.loc[data['근로자수'] >= 300, '기업규모'] = 4
data.loc[data['근로자수'] >= 1000, '기업규모'] = 5
#data.drop(columns = ["근로자수"], inplace = True)
data.head()

,연번,시도명,시군구명,관리기관,단지명,입주일,최초등록일,기준년월,노동지청,행정구역,...,평균급여액_2016,평균급여액_2017,평균급여액_2018,평균급여액_2019,평균급여액_2020,평균급여액_2021,평균급여액_2022,위도,경도,재해발생년도
0,164,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20180604.0,NaN,202012,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.481999,126.897686,2019
1,719,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20100202.0,20210419.0,201812,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482491,126.894409,2018
2,868,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,NaN,19970930.0,201912,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482035,126.897731,2019
3,892,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20060119.0,20060703.0,201912,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482128,126.895313,2018
4,979,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20030626.0,20111130.0,202112,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.482491,126.894409,2021


In [5]:
df = data[["업체명", "재해발생년도", "대업종", "매출액_2016", "매출액_2017", "매출액_2018", "매출액_2019", "매출액_2020", "매출액_2021", "기업규모"]]
df

,업체명,재해발생년도,대업종,매출액_2016,매출액_2017,매출액_2018,매출액_2019,매출액_2020,매출액_2021,기업규모
0,카플스,2019,기타의사업,0,0,930792,2259426,1515439,0,1
1,현정상사,2018,기타의사업,745100,0,0,0,0,0,1
2,모아전자,2019,제조업,0,0,0,0,0,0,0
3,맵시자수,2018,제조업,0,0,0,0,0,0,0
4,세광제록스,2021,기타의사업,1601829,1625162,1181916,1103319,1009248,0,1
...,...,...,...,...,...,...,...,...,...,...
15439,(주)케이조선,2017,제조업,1068181000,395828000,349783000,360496000,286890000,213246000,5
15440,(주)케이조선,2018,제조업,1068181000,395828000,349783000,360496000,286890000,213246000,5
15441,(주)케이조선,2019,제조업,1068181000,395828000,349783000,360496000,286890000,213246000,5
15442,(주)케이조선,2019,제조업,1068181000,395828000,349783000,360496000,286890000,213246000,5


In [6]:
df.매출액 = 0
for i, j in enumerate(df.재해발생년도.values, 0):
    if j == 2016:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2016"]
    elif j == 2017:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2017"]
    elif j == 2018:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2018"]
    elif j == 2019:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2019"]
    elif j == 2020:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2020"]
    elif j == 2021:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2021"]
    else:
        df.loc[i, "매출액"] = 0


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [7]:
df[df["매출액"] == 0]

,업체명,재해발생년도,대업종,매출액_2016,매출액_2017,매출액_2018,매출액_2019,매출액_2020,매출액_2021,기업규모,매출액
1,현정상사,2018,기타의사업,745100,0,0,0,0,0,1,0.0
2,모아전자,2019,제조업,0,0,0,0,0,0,0,0.0
3,맵시자수,2018,제조업,0,0,0,0,0,0,0,0.0
4,세광제록스,2021,기타의사업,1601829,1625162,1181916,1103319,1009248,0,1,0.0
7,한국산업기술시험원,2015,기타의사업,132738184,93988140,103797521,121974739,124268260,0,3,0.0
...,...,...,...,...,...,...,...,...,...,...,...
15306,(주)오리엔탈마린텍,2015,제조업,91727132,48027969,33635429,70584777,98690565,55529779,3,0.0
15337,(주)케이조선,2015,제조업,1068181000,395828000,349783000,360496000,286890000,213246000,5,0.0
15349,(주)케이조선,2015,제조업,1068181000,395828000,349783000,360496000,286890000,213246000,5,0.0
15369,(주)케이조선,1998,제조업,1068181000,395828000,349783000,360496000,286890000,213246000,5,0.0


In [8]:
grouped = df.groupby(by=['대업종', '기업규모'])
df_grouped = grouped.매출액.median().unstack()
df_grouped

기업규모,0,1,2,3,4,5
대업종,,,,,,
건설업,3528930.0,10509196.0,146080796.0,98670959.5,9.251875e+09,1.014844e+10
금융및보험업,NaN,0.0,NaN,NaN,NaN,NaN
기타의사업,1842913.5,5196785.0,18356749.5,252730858.0,0.000000e+00,1.756484e+09
운수·창고·통신업,0.0,18518676.0,31903090.5,484003153.0,3.367526e+07,NaN
전기·가스·증기및수도사업,NaN,23054669.0,369668907.0,659087578.5,NaN,NaN
제조업,547094.0,3806815.0,23707204.0,78308359.0,4.213202e+08,3.488273e+09


In [9]:
df[df["매출액"] == 0].대업종.value_counts()

제조업          2418
기타의사업         201
건설업            29
운수·창고·통신업      12
금융및보험업          1
Name: 대업종, dtype: int64

In [10]:
df_0_제조업 = df_grouped.iloc[5,0]
df_1_제조업 = df_grouped.iloc[5,1]
df_2_제조업 = df_grouped.iloc[5,2]
df_3_제조업 = df_grouped.iloc[5,3]
df_4_제조업 = df_grouped.iloc[5,4]
df_5_제조업 = df_grouped.iloc[5,5]

df_0_건설업 = df_grouped.iloc[0,0]
df_1_건설업 = df_grouped.iloc[0,1]
df_2_건설업 = df_grouped.iloc[0,2]
df_3_건설업 = df_grouped.iloc[0,3]
df_4_건설업 = df_grouped.iloc[0,4]
df_5_건설업 = df_grouped.iloc[0,5]

df_0_기타의사업 = df_grouped.iloc[2,0]
df_1_기타의사업 = df_grouped.iloc[2,1]
df_2_기타의사업 = df_grouped.iloc[2,2]
df_3_기타의사업 = df_grouped.iloc[2,3]
df_4_기타의사업 = 252730858.0
df_5_기타의사업 = df_grouped.iloc[2,5]

df_0_운수·창고·통신업 = 18518676.0
df_1_운수·창고·통신업 = df_grouped.iloc[3,1]
df_2_운수·창고·통신업 = df_grouped.iloc[3,2]
df_3_운수·창고·통신업 = df_grouped.iloc[3,3]
df_4_운수·창고·통신업 = df_grouped.iloc[3,4]
df_5_운수·창고·통신업 = df_grouped.iloc[3,5]

In [11]:
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_제조업

df.loc[(df.대업종 == '건설업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_건설업

df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_기타의사업

df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_운수·창고·통신업


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [12]:
df_매출액 = df[["매출액"]]
df_매출액

,매출액
0,2259426.0
1,5196785.0
2,547094.0
3,547094.0
4,5196785.0
...,...
15439,395828000.0
15440,349783000.0
15441,360496000.0
15442,360496000.0


In [13]:
data = pd.concat([data, df_매출액], axis = 1)
data

,연번,시도명,시군구명,관리기관,단지명,입주일,최초등록일,기준년월,노동지청,행정구역,...,평균급여액_2017,평균급여액_2018,평균급여액_2019,평균급여액_2020,평균급여액_2021,평균급여액_2022,위도,경도,재해발생년도,매출액
0,164,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20180604.0,NaN,202012,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,37.481999,126.897686,2019,2259426.0
1,719,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20100202.0,20210419.0,201812,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,37.482491,126.894409,2018,5196785.0
2,868,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,NaN,19970930.0,201912,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,37.482035,126.897731,2019,547094.0
3,892,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20060119.0,20060703.0,201912,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,37.482128,126.895313,2018,547094.0
4,979,서울특별시,구로구,한국산업단지공단 서울지역본부,서울디지털국가산업단지,20030626.0,20111130.0,202112,서울관악,서울,...,NaN,NaN,NaN,NaN,NaN,NaN,37.482491,126.894409,2021,5196785.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15439,60369,경상남도,창원시 진해구,한국산업단지공단 경남지역본부,진해국가산업단지,20090429.0,19921205.0,201712,창 원,경남,...,4006147.0,3424842.0,3661060.0,3620076.0,3625055.0,3602385.0,35.108088,128.716724,2017,395828000.0
15440,60369,경상남도,창원시 진해구,한국산업단지공단 경남지역본부,진해국가산업단지,20090429.0,19921205.0,202012,창 원,경남,...,4006147.0,3424842.0,3661060.0,3620076.0,3625055.0,3602385.0,35.108088,128.716724,2018,349783000.0
15441,60369,경상남도,창원시 진해구,한국산업단지공단 경남지역본부,진해국가산업단지,20090429.0,19921205.0,202012,창 원,경남,...,4006147.0,3424842.0,3661060.0,3620076.0,3625055.0,3602385.0,35.108088,128.716724,2019,360496000.0
15442,60369,경상남도,창원시 진해구,한국산업단지공단 경남지역본부,진해국가산업단지,20090429.0,19921205.0,202012,창 원,경남,...,4006147.0,3424842.0,3661060.0,3620076.0,3625055.0,3602385.0,35.108088,128.716724,2019,360496000.0


In [86]:
data.to_excel("C:/Users/PC/Desktop/데이터셋_005_민재3.xlsx", index = False)

In [29]:
df2 = data[["업체명", "대업종", "기업규모", "재해발생년도", "부채비율_2016", "부채비율_2017", "부채비율_2018", "부채비율_2019", "부채비율_2020", "부채비율_2021"]]
df2.head()

,업체명,대업종,기업규모,재해발생년도,부채비율_2016,부채비율_2017,부채비율_2018,부채비율_2019,부채비율_2020,부채비율_2021
0,카플스,기타의사업,1,2019,NaN,NaN,19.9379,221.8789,484.2362,NaN
1,현정상사,기타의사업,1,2018,26.2280,NaN,NaN,NaN,NaN,NaN
2,모아전자,제조업,0,2019,NaN,NaN,NaN,NaN,NaN,NaN
3,맵시자수,제조업,0,2018,NaN,NaN,NaN,NaN,NaN,NaN
4,세광제록스,기타의사업,1,2021,91.8907,56.4693,10.5195,10.4562,9.0936,NaN


In [30]:
print("2016 부채비율 null :", df2["부채비율_2016"].isnull().sum())
print("2017 부채비율 null :", df2["부채비율_2017"].isnull().sum())
print("2018 부채비율 null :", df2["부채비율_2018"].isnull().sum())
print("2019 부채비율 null :", df2["부채비율_2019"].isnull().sum())
print("2020 부채비율 null :", df2["부채비율_2020"].isnull().sum())
print("2021 부채비율 null :", df2["부채비율_2021"].isnull().sum())

2016 부채비율 null : 4771
2017 부채비율 null : 4422
2018 부채비율 null : 4067
2019 부채비율 null : 3777
2020 부채비율 null : 1611
2021 부채비율 null : 2652


In [31]:
df2.부채비율 = 0
for i, j in enumerate(df2.재해발생년도.values, 0):
    if j == 2016:
        df2.loc[i, "부채비율"] = df2.loc[i, "부채비율_2016"]
    elif j == 2017:
        df2.loc[i, "부채비율"] = df2.loc[i, "부채비율_2017"]
    elif j == 2018:
        df2.loc[i, "부채비율"] = df2.loc[i, "부채비율_2018"]
    elif j == 2019:
        df2.loc[i, "부채비율"] = df2.loc[i, "부채비율_2019"]
    elif j == 2020:
        df2.loc[i, "부채비율"] = df2.loc[i, "부채비율_2020"]
    elif j == 2021:
        df2.loc[i, "부채비율"] = df2.loc[i, "부채비율_2021"]
    else:
        df2.loc[i, "부채비율"] = 0


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [34]:
df2[df2["부채비율"] == 0]

,업체명,대업종,기업규모,재해발생년도,부채비율_2016,부채비율_2017,부채비율_2018,부채비율_2019,부채비율_2020,부채비율_2021,부채비율
7,한국산업기술시험원,기타의사업,3,2015,91.0591,105.7695,85.8712,70.9176,75.3097,NaN,0.0
113,(주)시그너스정보기술,기타의사업,1,2014,27.7822,23.7645,22.8491,26.6227,21.4253,21.3675,0.0
274,롯데정보통신(주),기타의사업,5,2015,NaN,NaN,58.1138,77.9804,56.5756,72.0652,0.0
419,(주)아쿠아픽,기타의사업,1,2014,186.3695,209.5601,493.0022,389.5102,372.4235,495.2454,0.0
431,해피랜드코퍼레이션(주),기타의사업,1,2014,145.3534,136.2985,141.9556,189.7983,69.4997,81.1586,0.0
...,...,...,...,...,...,...,...,...,...,...,...
15306,(주)오리엔탈마린텍,제조업,3,2015,-953.8785,-679.3354,-572.9745,-2277.8862,430.3744,308.3865,0.0
15337,(주)케이조선,제조업,5,2015,105.1325,87.2014,100.2536,87.0389,134.6134,196.2943,0.0
15349,(주)케이조선,제조업,5,2015,105.1325,87.2014,100.2536,87.0389,134.6134,196.2943,0.0
15369,(주)케이조선,제조업,5,1998,105.1325,87.2014,100.2536,87.0389,134.6134,196.2943,0.0
